# C11-neural-training — Session 2: Manual Backpropagation

*One 90-minute session. Prerequisites: F4's local derivatives and multivariable
chain rule, C5's ReLU MLP forward pass, and Session 1's stable cross-entropy
gradient.*

**Learning contract.** Backpropagation is organized chain rule. We will draw a
dependency graph, process it in reverse dependency order, add contributions at
branches, and derive every gradient of a two-layer MLP with a complete shape
ledger.


In [ ]:
import numpy as np

SEED = 20260804
ATOL = 1e-7
RTOL = 1e-6
rng = np.random.default_rng(SEED)

## 1. Computation graphs and local derivatives

A **computation graph** records values as nodes and elementary operations as
edges. For $u=xw$, $v=u+b$, and $L=v^2$:

- local derivative $\partial u/\partial w=x$;
- local derivative $\partial v/\partial u=1$;
- local derivative $\partial L/\partial v=2v$.

Reverse mode begins at $\partial L/\partial L=1$ and multiplies local
derivatives along each backward edge:
$\partial L/\partial w=(2v)(1)(x)$.

The dependency order is not optional. A node may send a gradient backward only
after all downstream contributions to that node have arrived.

**Checkpoint 1A.** For $x=3,w=2,b=-1$, compute $u,v,L$ and
$\partial L/\partial w$.

**Checkpoint 1B.** Why does a forward topological order become a reverse
topological order during backpropagation?


In [ ]:
x, w, b = 3.0, 2.0, -1.0
u = x * w
v = u + b
L = v**2
dL_dw = 2 * v * x
print((u, v, L, dL_dw))
assert np.isclose(dL_dw, 30.0, atol=ATOL, rtol=RTOL)

## 2. Branches require gradient accumulation

Suppose one value $a$ is used twice: $q=a^2+a$. The graph has two paths from
$a$ to $q$. The multivariable chain rule adds them:

$$\frac{dq}{da}=\underbrace{2a}_{a^2\text{ path}}
+\underbrace{1}_{a\text{ path}}.$$

This is **gradient accumulation**: gradients at a shared node are sums, never
an arbitrary last write. In neural networks, a parameter is shared across all
examples in a batch, so its gradient accumulates contributions from every
example. Broadcasting a bias across $N$ rows similarly produces a sum over
axis 0 in the backward pass.

**Checkpoint 2A.** At $a=4$, what are the two contributions and their sum?

**Checkpoint 2B.** If $B$ of shape $(H,)$ broadcasts into $Z$ of shape
$(N,H)$, what is the shape and formula of $\partial L/\partial B$ given
$G=\partial L/\partial Z$?


In [ ]:
a = 4.0
path_square = 2 * a
path_identity = 1.0
total = path_square + path_identity
print(path_square, path_identity, total)
assert np.isclose(total, 9.0, atol=ATOL, rtol=RTOL)

## 3. Two-layer MLP: forward graph and shape ledger

Define:

- $X\in\mathbb R^{N\times D}$: $N$ examples, $D$ input features;
- $W_1\in\mathbb R^{H\times D}$ and $b_1\in\mathbb R^H$: hidden layer
  with $H$ units;
- $W_2\in\mathbb R^{C\times H}$ and $b_2\in\mathbb R^C$: output layer
  with $C$ classes.

The C5 row-is-a-unit convention gives

$$Z_1=XW_1^\top+b_1\quad(N,H),$$
$$A_1=\operatorname{ReLU}(Z_1)\quad(N,H),$$
$$Z_2=A_1W_2^\top+b_2\quad(N,C),$$
$$L=\operatorname{CE}(Z_2,y)\quad\text{scalar}.$$

A **cache** stores $X,Z_1,A_1,Z_2,P,y$ from the forward pass. Backward formulas
must use the exact values that produced the loss, not recomputed values after
parameters move.

At $Z_1=0$, this course pins $\operatorname{ReLU}'(0)=0$.

**Checkpoint 3A.** For $N=5,D=2,H=4,C=3$, state all six array shapes.

**Checkpoint 3B.** Why may parameters not be updated before the whole backward
pass is complete?


In [ ]:
def stable_softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / exp_shifted.sum(axis=1, keepdims=True)

def forward_two_layer(X, W1, b1, W2, b2, y):
    Z1 = X @ W1.T + b1
    A1 = np.maximum(Z1, 0.0)
    Z2 = A1 @ W2.T + b2
    P = stable_softmax(Z2)
    loss = -np.log(P[np.arange(X.shape[0]), y]).mean()
    return loss, (X, Z1, A1, Z2, P, y)

X = np.array([[1.0, -2.0]])
W1 = np.array([[1.0, -1.0], [0.5, 1.0]])
b1 = np.zeros(2)
W2 = np.array([[1.0, -1.0], [-1.0, 1.0]])
b2 = np.zeros(2)
y = np.array([1])
loss, cache = forward_two_layer(X, W1, b1, W2, b2, y)
print("loss:", loss)
print("shapes:", [value.shape for value in cache])

## 4. Complete backward derivation

Start from Session 1:

$$G_2=\frac{\partial L}{\partial Z_2}=\frac{P-Y}{N}
\quad(N,C).$$

For $Z_{2,ic}=\sum_h A_{1,ih}W_{2,ch}+b_{2,c}$,

$$\frac{\partial L}{\partial W_2}=G_2^\top A_1\quad(C,H),$$
$$\frac{\partial L}{\partial b_2}=\sum_i G_{2,i:}\quad(C),$$
$$\frac{\partial L}{\partial A_1}=G_2W_2\quad(N,H).$$

Through ReLU, define mask $M=\mathbf1[Z_1>0]$ of shape $(N,H)$:

$$G_1=\frac{\partial L}{\partial Z_1}
=\frac{\partial L}{\partial A_1}\odot M\quad(N,H).$$

Finally,

$$\frac{\partial L}{\partial W_1}=G_1^\top X\quad(H,D),$$
$$\frac{\partial L}{\partial b_1}=\sum_i G_{1,i:}\quad(H),$$
$$\frac{\partial L}{\partial X}=G_1W_1\quad(N,D).$$

Every matrix product is now forced by shapes. The bias gradients are sums
because broadcasting created one use per example.

**Checkpoint 4A.** Explain why $G_2^\top A_1$, not $A_1^\top G_2$, has
the shape of $W_2$.

**Checkpoint 4B.** If one hidden pre-activation is negative, what happens to
the gradient entering that hidden unit for that example?


In [ ]:
def backward_two_layer(cache, W1, W2):
    X, Z1, A1, _Z2, P, y = cache
    n = X.shape[0]
    G2 = P.copy()
    G2[np.arange(n), y] -= 1.0
    G2 /= n
    dW2 = G2.T @ A1
    db2 = G2.sum(axis=0)
    dA1 = G2 @ W2
    G1 = dA1 * (Z1 > 0.0)
    dW1 = G1.T @ X
    db1 = G1.sum(axis=0)
    dX = G1 @ W1
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "X": dX}

grads = backward_two_layer(cache, W1, W2)
for name, grad in grads.items():
    print(name, grad.shape, grad)

## 5. Worked example: one complete backward pass

For the values in the code above:

1. $Z_1=(3,-1.5)$, so $A_1=(3,0)$ and mask $M=(1,0)$.
2. $Z_2=(3,-3)$. Let
   $q=e^3/(e^3+e^{-3})=1/(1+e^{-6})$.
   Since target class is 1, $G_2=(q, -q)$.
3. $dW_2=G_2^\top A_1=
   \begin{pmatrix}3q&0\\-3q&0\end{pmatrix}$ and
   $db_2=(q,-q)$.
4. $dA_1=G_2W_2=(2q,-2q)$.
   Applying the ReLU mask gives $G_1=(2q,0)$.
5. $dW_1=G_1^\top X=
   \begin{pmatrix}2q&-4q\\0&0\end{pmatrix}$ and
   $db_1=(2q,0)$.
6. $dX=G_1W_1=(2q,-2q)$.

The inactive hidden unit receives zero gradient in this example, but its
weights are not globally frozen: a different example may activate it, and
batch contributions accumulate.

**Checkpoint 5A.** Which gradient arrays contain an entire zero row, and why?

**Checkpoint 5B.** If the batch had two examples, where would the factor
$1/2$ first enter this derivation?


In [ ]:
q = 1.0 / (1.0 + np.exp(-6.0))
expected = {
    "W2": np.array([[3*q, 0.0], [-3*q, 0.0]]),
    "b2": np.array([q, -q]),
    "W1": np.array([[2*q, -4*q], [0.0, 0.0]]),
    "b1": np.array([2*q, 0.0]),
    "X": np.array([[2*q, -2*q]]),
}
for name, expected_value in expected.items():
    assert np.allclose(grads[name], expected_value, atol=ATOL, rtol=RTOL), name
print("all hand-derived arrays match")

## 6. Gradient checking as an audit

For a scalar parameter coordinate $\theta$, the centered finite difference

$$g_{\text{num}}=
\frac{L(\theta+\varepsilon)-L(\theta-\varepsilon)}{2\varepsilon}$$

approximates $\partial L/\partial\theta$. It is an audit, not a training
method: it costs two forward passes per coordinate and suffers roundoff if
$\varepsilon$ is too small. Use a smooth point away from ReLU's kink and
compare with named tolerances.

**Checkpoint 6A.** Why is a centered difference generally preferable to
$(L(\theta+\varepsilon)-L(\theta))/\varepsilon$?

**Checkpoint 6B.** Why can a correct ReLU backward pass fail a finite
difference check exactly at $Z_1=0$?


In [ ]:
epsilon = 1e-6
row, col = 0, 1
W1_plus = W1.copy(); W1_plus[row, col] += epsilon
W1_minus = W1.copy(); W1_minus[row, col] -= epsilon
loss_plus, _ = forward_two_layer(X, W1_plus, b1, W2, b2, y)
loss_minus, _ = forward_two_layer(X, W1_minus, b1, W2, b2, y)
numeric = (loss_plus - loss_minus) / (2 * epsilon)
analytic = grads["W1"][row, col]
print("numeric:", numeric, "analytic:", analytic)
assert np.isclose(numeric, analytic, atol=ATOL, rtol=RTOL)

## 7. Common pitfalls, exam connections, and forward links

- **Overwrite at a branch:** `grad = contribution` loses earlier paths.
  Fix: initialize to zero and add every downstream contribution.
- **Transpose by memory:** formulas copied without shapes often return
  transposed parameter gradients. Fix: demand that $dW_\ell$ match $W_\ell$.
- **Bias mean instead of sum:** averaging twice shrinks gradients. The $1/N$
  already entered at $G_2$; bias broadcasting reverses with a sum.
- **Recomputing after update:** a mixed-version cache no longer differentiates
  the reported loss. Complete backward first, then update every parameter.
- **ReLU boundary ambiguity:** state the convention; this course uses zero.

Round 1 derivation problems often ask only one entry of one gradient.
Build the whole dependency/shape path first, then evaluate the requested
entry. Session 3 turns these equations into a complete NumPy trainer; Session
4 compares them with autograd.

**Checkpoint 7A.** A returned `dW2` has shape $(H,C)$. Name the likely
transpose error.

**Checkpoint 7B.** A programmer divides `db2` by $N$ after using
$(P-Y)/N$. By what factor is the result wrong?


## Checkpoint answers

**1A.** $u=6,v=5,L=25,dL/dw=30$. **1B.** A node's gradient depends on all
operations that consume it, so those downstream gradients must exist first.

**2A.** $8$ and $1$, total $9$. **2B.**
$dB=\sum_{i=1}^N G_{i:}$, shape $(H,)$.

**3A.** $X(5,2),W_1(4,2),b_1(4,),W_2(3,4),b_2(3,),Z_1/A_1(5,4),
Z_2/P(5,3)$; the six forward arrays are $X,Z_1,A_1,Z_2,P,y$ with
$y(5,)$. **3B.** Earlier-layer gradients use the forward-version later-layer
weights; moving them early mixes parameter versions.

**4A.** $(C,N)(N,H)=(C,H)$, exactly $W_2$; the alternative is $(H,C)$.
**4B.** The ReLU mask multiplies it by zero.

**5A.** The second rows of $dW_1$ (and its $db_1$ entry) are zero because the
second hidden unit is inactive; the second columns of $dW_2$ are zero because
its activation is zero. **5B.** In $G_2=(P-Y)/2$.

**6A.** Its leading truncation error cancels, giving a more accurate local
slope for the same scale. **6B.** ReLU is not differentiable there; the
symmetric numerical slope need not equal the pinned one-sided convention.

**7A.** It computed $A_1^\top G_2$ instead of $G_2^\top A_1$.
**7B.** It is too small by another factor $N$.
